In [ ]:
pip install feedparser requests beautifulsoup4

: 

In [ ]:
import pandas as pd

RSS einlesen

In [ ]:
import feedparser

RSS_URL = "https://correctiv.org/feed/"

feed = feedparser.parse(RSS_URL)

print("entries:", len(feed.entries))


Einträge extrahieren

In [ ]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)


In [ ]:
print(rss_items)

Volltext aus den Artikeln holen

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

RATING_LABELS = [
    "FEHLENDER KONTEXT",
    "TEILWEISE FALSCH",
    "FALSCH",
    "RICHTIG",
    "UNBELEGT",
    "IRREFÜHREND",
]

def extract_box(article, anchor):
    pattern = re.compile(rf"\b{re.escape(anchor)}\b", re.I)
    for node in article.find_all(string=pattern):
        if not node or not node.strip():
            continue
        parent = node.parent
        if parent.name in ("script", "style"):
            continue

        container = parent
        while container and container != article and container.name not in ("section", "aside", "div"):
            container = container.parent

        if container is None:
            continue
        if container == article:
            container = parent

        box_text = " ".join(container.stripped_strings)
        if box_text:
            return box_text

    return None

def extract_rating_label(rating_box):
    if not rating_box:
        return None

    upper = rating_box.upper()
    for label in RATING_LABELS:
        if label in upper:
            return label
    return None

def fetch_article_v2(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return {
            "text": None,
            "claim_box": None,
            "rating_box": None,
            "rating_label": None,
        }

    text = "\n".join(s for s in article.stripped_strings)

    claim_box = extract_box(article, "BEHAUPTUNG")
    rating_box = extract_box(article, "BEWERTUNG")
    rating_label = extract_rating_label(rating_box)

    return {
        "text": text,
        "claim_box": claim_box,
        "rating_box": rating_box,
        "rating_label": rating_label,
    }


In [ ]:
results = []

for item in rss_items:
    try:
        article_data = fetch_article_v2(item["url"])
        item.update(article_data)
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)


In [ ]:
import json

with open("../outputs/correctiv_articles_v2.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


## Themenspezifisches Scraping

In [ ]:
import time
import requests
import feedparser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RSS_URL = "https://correctiv.org/faktencheck/tag/klima/feed/"

def make_session():
    s = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    return s

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/121.0 Safari/537.36",
    "Accept": "application/rss+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}

session = make_session()

resp = session.get(RSS_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

feed = feedparser.parse(resp.text)

print("HTTP:", resp.status_code)
print("entries:", len(feed.entries))
print(feed.entries[0].get("title"), feed.entries[0].get("link"))


In [ ]:
def parse_rss(feed):
    items = []
    for e in feed.entries:
        items.append({
            "title": e.get("title"),
            "url": e.get("link"),
            "published": e.get("published"),
            "author": e.get("author"),
            "categories": [t["term"] for t in e.get("tags", [])],
            "summary": e.get("summary"),
        })
    return items

rss_items = parse_rss(feed)

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (ThesisScraper/1.0; academic research)"
}

RATING_LABELS = [
    "FEHLENDER KONTEXT",
    "TEILWEISE FALSCH",
    "FALSCH",
    "RICHTIG",
    "UNBELEGT",
    "IRREFÜHREND",
]

def extract_box(article, anchor):
    pattern = re.compile(rf"\b{re.escape(anchor)}\b", re.I)
    for node in article.find_all(string=pattern):
        if not node or not node.strip():
            continue
        parent = node.parent
        if parent.name in ("script", "style"):
            continue

        container = parent
        while container and container != article and container.name not in ("section", "aside", "div"):
            container = container.parent

        if container is None:
            continue
        if container == article:
            container = parent

        box_text = " ".join(container.stripped_strings)
        if box_text:
            return box_text

    return None

def extract_rating_label(rating_box):
    if not rating_box:
        return None

    upper = rating_box.upper()
    for label in RATING_LABELS:
        if label in upper:
            return label
    return None

def fetch_article_v2(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.select_one("article")
    if not article:
        return {
            "text": None,
            "claim_box": None,
            "rating_box": None,
            "rating_label": None,
        }

    text = "\n".join(s for s in article.stripped_strings)

    claim_box = extract_box(article, "BEHAUPTUNG")
    rating_box = extract_box(article, "BEWERTUNG")
    rating_label = extract_rating_label(rating_box)

    return {
        "text": text,
        "claim_box": claim_box,
        "rating_box": rating_box,
        "rating_label": rating_label,
    }


In [ ]:
results = []

for item in rss_items:
    try:
        article_data = fetch_article_v2(item["url"])
        item.update(article_data)
        results.append(item)
        time.sleep(1.2)  # wichtig!
    except Exception as e:
        item["error"] = str(e)
        results.append(item)


In [ ]:
import json

with open("../outputs/correctiv_articles_klima_v2.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


Look at scraped articles

In [ ]:
path = "../outputs/correctiv_articles_klima_v2.jsonl"
df = pd.read_json(path, lines=True)

df.head(10)


In [ ]:
import textwrap
from IPython.display import display, Markdown

df = pd.read_json("../outputs/correctiv_articles_klima_v2.jsonl", lines=True)

row = None
if "rating_label" in df.columns:
    matches = df[df["rating_label"] == "FEHLENDER KONTEXT"]
    if len(matches) > 0:
        row = matches.iloc[0]

if row is None:
    row = df.iloc[0]  # fallback

title = row.get("title", "Untitled")
url = row.get("url", "")
text = row.get("text", "")
claim_box = row.get("claim_box", "")
rating_box = row.get("rating_box", "")
rating_label = row.get("rating_label", None)

display(Markdown(f"## {title}\n\n{url}"))
print("rating_label:", rating_label)
print("\nclaim_box (excerpt):")
print((claim_box or "")[:1000])
print("\nrating_box (excerpt):")
print((rating_box or "")[:1000])
print("\ntext (excerpt):")
print((text or "")[:1000])


In [ ]:
# Optional debugging: only if Bewertung/Behauptung boxes are missing
import re
import requests
from bs4 import BeautifulSoup

try:
    _row = row
except NameError:
    _row = None

if _row is not None and not _row.get("rating_box"):
    url = _row.get("url")
    if url:
        r = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")
        article = soup.select_one("article")
        if article:
            node = article.find(string=re.compile("Bewertung", re.I))
            if node:
                container = node.parent
                snippet = container.prettify()
                print(snippet[:2000])